# Deployment & Demo - Model Serialization and Live Prediction

This notebook demonstrates:
- Loading the best trained model
- Making predictions on new data
- Model versioning and documentation
- Deployment-ready code patterns
- Live prediction demo with sample conversations

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from datetime import datetime
from IPython.display import Markdown, display
import json

# Setup path
PROJECT_ROOT = Path('.').resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preparation import load_hf_dataset, dataset_to_dataframe
from src.feature_engineering import SalesFeatureEngineer, FeatureEngineeringConfig

sns.set_theme(style='whitegrid', context='notebook')

print('Deployment environment ready')

## 1. Load Production Models and Components

In [ ]:
# Load best model and supporting components
best_model = joblib.load('../results/model_best.pkl')
pca_model = joblib.load('../results/pca_model.pkl')
feature_selector = joblib.load('../results/feature_selector.pkl')
selected_features = joblib.load('../results/selected_features.pkl')

# Load evaluation metrics
with open('../metrics/evaluation_metrics.json', 'r') as f:
    eval_metrics = json.load(f)

print('✓ Loaded best model')
print('✓ Loaded PCA transformer')
print('✓ Loaded feature selector')
print('✓ Production environment ready')
print(f'\nModel type: {type(best_model).__name__}')

## 2. Create Prediction Pipeline

In [ ]:
class SalesPredictionPipeline:
    """
    Production-ready prediction pipeline for SaaS sales conversations.
    
    Usage:
        pipeline = SalesPredictionPipeline(model, pca, selector)
        predictions = pipeline.predict_batch(dataframe)
        result = pipeline.predict_single(row)
    """
    
    def __init__(self, model, pca, selector, fe_config=None):
        self.model = model
        self.pca = pca
        self.selector = selector
        self.fe_config = fe_config or FeatureEngineeringConfig(text_strategy='embeddings')
        self.feature_engineer = SalesFeatureEngineer(self.fe_config)
        self.created_at = datetime.now()
    
    def preprocess(self, df):
        """
        Transform raw data through feature engineering and dimensionality reduction.
        """
        # Feature engineering
        X_engineered, _ = self.feature_engineer.fit_transform(df)
        
        # Feature selection
        X_selected = self.selector.transform(X_engineered)
        
        # PCA transformation
        X_selected_dense = X_selected.toarray() if hasattr(X_selected, 'toarray') else X_selected
        X_pca = self.pca.transform(X_selected_dense)
        
        return X_pca
    
    def predict_proba(self, df):
        """
        Get probability predictions for each class.
        
        Returns:
            ndarray: shape (n_samples, 2) with probabilities for [Lost, Won]
        """
        X_processed = self.preprocess(df)
        return self.model.predict_proba(X_processed)
    
    def predict(self, df, threshold=0.5):
        """
        Get binary predictions (0=Lost, 1=Won).
        
        Args:
            df: Input data
            threshold: Decision threshold (default 0.5)
        
        Returns:
            ndarray: Binary predictions (0 or 1)
        """
        proba = self.predict_proba(df)
        return (proba[:, 1] >= threshold).astype(int)
    
    def predict_with_confidence(self, df, threshold=0.5):
        """
        Get predictions with confidence scores and labels.
        
        Returns:
            DataFrame with columns: prediction, probability_won, probability_lost, confidence
        """
        proba = self.predict_proba(df)
        predictions = (proba[:, 1] >= threshold).astype(int)
        confidence = np.max(proba, axis=1)
        
        result_df = pd.DataFrame({
            'prediction': ['Won' if p == 1 else 'Lost' for p in predictions],
            'probability_won': proba[:, 1],
            'probability_lost': proba[:, 0],
            'confidence': confidence,
            'threshold': threshold
        })
        
        return result_df


# Initialize pipeline
pipeline = SalesPredictionPipeline(
    model=best_model,
    pca=pca_model,
    selector=feature_selector
)

print('✓ Production pipeline initialized')
print(f'  Created: {pipeline.created_at}')

## 3. Model Information Card

In [ ]:
# Load comparison results to identify best model
comparison_df = pd.read_csv('../metrics/model_comparison.csv', index_col=0)
best_model_name = comparison_df['f1'].idxmax()
best_metrics = comparison_df.loc[best_model_name]

# Load evaluation metrics
eval_df = pd.read_csv('../metrics/evaluation_metrics.csv', index_col=0)
best_eval = eval_df.loc[best_model_name]

info_text = f"""
## Production Model Information

### Model Identity
- **Type**: {best_model_name}
- **Version**: 1.0
- **Created**: {pipeline.created_at.strftime('%Y-%m-%d %H:%M:%S')}
- **Status**: Ready for Production

### Training Metrics
- **Accuracy**: {best_metrics['accuracy']:.4f}
- **F1 Score**: {best_metrics['f1']:.4f}
- **ROC-AUC**: {best_metrics['roc_auc']:.4f}
- **Training Time**: {best_metrics['training_time']:.2f}s

### Evaluation Metrics (Test Set)
- **Accuracy**: {best_eval['Accuracy']:.4f}
- **Precision**: {best_eval['Precision']:.4f}
- **Recall**: {best_eval['Recall']:.4f}
- **F1**: {best_eval['F1']:.4f}
- **ROC-AUC**: {best_eval['ROC-AUC']:.4f}

### Pipeline Configuration
- **Feature Engineering**: Embeddings + Numerical Features
- **Feature Selection**: Top 500 features (f-statistic)
- **Dimensionality Reduction**: PCA ({pca_model.n_components_} components)
- **Decision Threshold**: 0.5 (configurable)

### Input Requirements
- **Format**: Pandas DataFrame
- **Required Columns**: Same as training set
- **Sample Size**: Batch or single predictions supported

### Output Format
- **Prediction**: Binary (0=Lost, 1=Won)
- **Confidence**: Probability estimates
- **Output Type**: Pandas Series/DataFrame
"""

display(Markdown(info_text))

In [ ]:
# Save model card
model_card = {
    'name': best_model_name,
    'version': '1.0',
    'created': pipeline.created_at.isoformat(),
    'status': 'production',
    'task': 'sales_outcome_prediction',
    'model_type': str(type(best_model).__name__),
    'training_metrics': best_metrics.to_dict(),
    'evaluation_metrics': best_eval.to_dict(),
    'pipeline_config': {
        'feature_engineering': 'embeddings',
        'feature_selection': 'top_500_f_statistic',
        'dimensionality_reduction': f'pca_{pca_model.n_components_}_components',
        'decision_threshold': 0.5
    }
}

with open('../results/model_card.json', 'w') as f:
    json.dump(model_card, f, indent=2)

print('✓ Saved model card to results/model_card.json')

In [ ]:
## 4. Live Demo - Batch Predictions

In [ ]:
# Load fresh data for demo
print('Loading demo dataset...')
raw_data = load_hf_dataset('DeepMostInnovations/saas-sales-conversations')
df_all = dataset_to_dataframe(raw_data)

# Take a random sample for demo
np.random.seed(42)
demo_indices = np.random.choice(len(df_all), size=20, replace=False)
df_demo = df_all.iloc[demo_indices].copy()
y_demo_true = df_demo['outcome'].astype(int).values

# Make predictions
print(f'\nMaking predictions on {len(df_demo)} samples...')
results = pipeline.predict_with_confidence(df_demo)

# Add true labels for comparison
results['actual'] = ['Won' if y == 1 else 'Lost' for y in y_demo_true]
results['correct'] = (results['prediction'] == results['actual']).astype(int)

print('\n=== LIVE PREDICTION RESULTS (Sample of 20) ===')
print(results[['actual', 'prediction', 'probability_won', 'confidence', 'correct']].to_string())

accuracy = results['correct'].mean()
print(f'\n✓ Accuracy on demo set: {accuracy:.2%}')

# Save demo results
results.to_csv('../results/deployment_demo_predictions.csv', index=False)
print('✓ Saved demo results to results/deployment_demo_predictions.csv')

In [ ]:
## 5. Confidence Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confidence distribution
axes[0].hist(results['confidence'], bins=20, edgecolor='black', alpha=0.7)
axes[0].axvline(results['confidence'].mean(), color='r', linestyle='--', label=f'Mean: {results["confidence"].mean():.3f}')
axes[0].set_xlabel('Confidence')
axes[0].set_ylabel('Count')
axes[0].set_title('Prediction Confidence Distribution (Demo)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Confidence by correctness
correct_conf = results[results['correct'] == 1]['confidence']
incorrect_conf = results[results['correct'] == 0]['confidence']

axes[1].hist([correct_conf, incorrect_conf], label=['Correct', 'Incorrect'], bins=15, edgecolor='black')
axes[1].set_xlabel('Confidence')
axes[1].set_ylabel('Count')
axes[1].set_title('Confidence by Prediction Correctness')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/deployment_confidence_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Saved confidence analysis to figures/deployment_confidence_analysis.png')

In [ ]:
## 6. Deployment Summary

In [ ]:
summary_text = f"""
## Deployment Summary

### ✓ Ready for Production

**Model**: {best_model_name}
- Training F1: {best_metrics['f1']:.4f}
- Test Accuracy: {best_eval['Accuracy']:.4f}
- Test F1: {best_eval['F1']:.4f}

### Usage Example

```python
# Load pipeline
pipeline = joblib.load('model_best_pipeline.pkl')

# Batch predictions
results = pipeline.predict_with_confidence(dataframe)

# Or directly
predictions = pipeline.predict(dataframe)
probabilities = pipeline.predict_proba(dataframe)
```

### Files Generated
- ✓ `model_best.pkl` - Trained model
- ✓ `model_card.json` - Model metadata
- ✓ `pca_model.pkl` - Feature transformer
- ✓ `feature_selector.pkl` - Feature selector
- ✓ `deployment_demo_predictions.csv` - Sample predictions
- ✓ Confidence analysis visualization

### Next Steps for Production
1. **API Development**: Wrap pipeline in REST/GraphQL API
2. **Monitoring**: Track prediction distributions and drift
3. **Versioning**: Implement model versioning strategy
4. **A/B Testing**: Compare against baseline in production
5. **Retraining**: Schedule monthly retraining on new data
6. **Documentation**: Generate API docs and user guides
7. **Compliance**: Audit for fairness and bias
8. **Performance**: Monitor latency and accuracy metrics
"""

display(Markdown(summary_text))

# Save pipeline for production use
joblib.dump(pipeline, '../results/model_best_pipeline.pkl')
print('\n✓ Saved production pipeline to results/model_best_pipeline.pkl')
print('✓ All deployment artifacts ready')